# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


We have one row per content for each date on the month of march

In [2]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

relevant = [f for f in files if "fact_content_daily_performance" in f]
for f in relevant:
    print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [3]:
from datasets import load_dataset

ds_march = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=hf_token
)

df_march = ds_march.to_pandas()
print(df_march.shape)
print(df_march['report_date'].min(), df_march['report_date'].max())

df_march_slim = df_march[['report_date', 'content_hash_id']]

daily_check = df_march_slim.groupby('report_date')['content_hash_id'].agg(
    total_rows='count',
    unique_ids='nunique'
).reset_index()

mismatches = daily_check[daily_check['total_rows'] != daily_check['unique_ids']]
print(f"Dates with duplicate content_hash_id: {len(mismatches)}")

print("Therefore we have one row per content(page), for each day on the month of March")



(9841378, 30)
2026-03-01 2026-03-31
Dates with duplicate content_hash_id: 0
Therefore we have one row per content(page), for each day on the month of March


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**How I'm picking features**for each one I ask:
1. Visibility: does it describe (or help derive) how visible the page is?
2. Performance: does it describe (or help derive) how well the page engages people?
3. Trend direction: does it describe (or help derive) which way things are moving over time?
4. Market opportunity: does it describe (or help derive) how hard the page is to rank for?
5. Content investment:  is it an attribute of the content itself that feeds into performance/visibility?

I don't need a feature for every single one of these just picking the strongest 5 overall.

**My 5 features:**
- **gsc_impressions** (visibility)
- **gsc_avg_position**  (visibility)
- **ctr** (derived from gsc_clicks / gsc_impressions) performance
- **trend_direction** (derived from report_date, comparing early vs late month) trend
- **word_count** (content investment)

**Label:**
No real labelthis is unsupervised. The archetypes are names I assign to
clusters after the fact, based on where they sit across visibility/performance/
trend. Not something pulled from a column.

**Constraints:**
**report_date** isn't a feature on its own, but I use it to derive **trend_direction**
by comparing early-month vs late-month values within March.

**Excluded (and why):**
- **content_hash_id, client_hash_id, keyword_hash_id, url_hash_id** these
  are hashed for privacy, so they carry no real meaning as features. I only use
  them for joins/grouping, never as signal.
- GA4 columns **ga4_\*,sessions_\*, `ai_\*, scroll_events**  GA4 availability
  is only ~4.2% of March, way too sparse to build anything reliable on. **scroll_events**
  gets cut for the same reason since it's a GA4-tracked interaction and won't fire
  without GA4 data present.
- **search_volume,competition,competition_level, cpc** these are real
  market-opportunity signals and I considered using them, but I'm capped at 5
  features and visibility/performance/trend/content-investment felt like the
  core of what defines an archetype. Cutting market-opportunity for scope, not
  because it's irrelevant.
- **keyword_token_count**close call against **word_count**, but token count is
  more about the keyword being targeted (search intent) than the content itself.
  **word_count** is a more direct signal of the actual investment put into the page.
- **gsc_click** kept as raw signal underneath **ctr**, but not counted separately
  since it's already folded into the derived metric.




## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

dim_files = [f for f in files if "dim_content" in f]
for f in dim_files:
    print(f)

dim_content.parquet


In [5]:
ds_content = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="dim_content.parquet",
    split="train",
    token=hf_token
)

df_content = ds_content.to_pandas()


In [6]:
df_march_joined = df_march.merge(
    df_content,
    on='content_hash_id',
    how='left',
    validate='m:1'
)

In [7]:
import gc
print("joined data size", df_march.shape[0], "un-joined data size", df_march_joined.shape[0])
print("same size we can continue")
print("filter using TRUE for either of them and both")
df_ga4_gsc_march = df_march_joined[(df_march_joined['gsc_data_available'] == True)& (df_march_joined['ga4_data_available'] == True)]
df_ga4_march = df_march_joined[(df_march_joined['ga4_data_available'] == True)]
df_gsc_march = df_march_joined[df_march_joined['gsc_data_available'] == True]
print("both ga4 and gsc availability",(df_ga4_gsc_march.shape[0]/ df_march_joined.shape[0]) * 100)
print("ga4 availability",(df_ga4_march.shape[0]/ df_march_joined.shape[0]) * 100)
print("gsc avialability", (df_gsc_march.shape[0]/ df_march_joined.shape[0]) * 100)

gsc_available = df_march[df_march['gsc_data_available'] == True]
print(f"Rows where gsc_data_available IS TRUE: {gsc_available.shape[0]:,} out of {df_march.shape[0]:,} ({gsc_available.shape[0]/df_march.shape[0]:.1%})")



joined data size 9841378 un-joined data size 9841378
same size we can continue
filter using TRUE for either of them and both
both ga4 and gsc availability 3.7021949568444583
ga4 availability 4.206382480177065
gsc avialability 36.69263592964319
Rows where gsc_data_available IS TRUE: 3,611,061 out of 9,841,378 (36.7%)


In [8]:
##Granularity
df_march_grain = df_march_joined[['report_date', 'content_hash_id']]

grain = df_march_grain.groupby('report_date')['content_hash_id'].agg(
    total_rows='count',
    unique_ids='nunique'
).reset_index()
print(grain)
print("Data has low granularity, Content is aggregated daily")

   report_date  total_rows  unique_ids
0   2026-03-01      275874      275874
1   2026-03-02      276269      276269
2   2026-03-03      311676      311676
3   2026-03-04      311675      311675
4   2026-03-05      311676      311676
5   2026-03-06      312187      312187
6   2026-03-07      312387      312387
7   2026-03-08      313374      313374
8   2026-03-09      313874      313874
9   2026-03-10      314047      314047
10  2026-03-11      314232      314232
11  2026-03-12      317742      317742
12  2026-03-13      317900      317900
13  2026-03-14      319584      319584
14  2026-03-15      319758      319758
15  2026-03-16      319924      319924
16  2026-03-17      320048      320048
17  2026-03-18      320682      320682
18  2026-03-19      323054      323054
19  2026-03-20      323738      323738
20  2026-03-21      324296      324296
21  2026-03-22      324476      324476
22  2026-03-23      324899      324899
23  2026-03-24      325117      325117
24  2026-03-25      32560

In [9]:
print(df_march['scroll_events'].isna().sum() / len(df_march))
print((df_march['scroll_events'] == 0).sum() / len(df_march))

0.30673966592889734
0.6807580198626656


In [10]:
#Count
print(f"Total rows in March 2026 partition: {df_march.shape[0]:,}")
print(f"Date range: {df_march['report_date'].min()} to {df_march['report_date'].max()}")
print(f"Number of distinct days: {df_march['report_date'].nunique()}")

Total rows in March 2026 partition: 9,841,378
Date range: 2026-03-01 to 2026-03-31
Number of distinct days: 31


## 4. Data limits

What this data can't tell you, based on what I actually checked:

**GA4 is basically unusable for this lane.** Only 4.2% of March rows have GA4
data available. That means for over 95% of content, I have zero idea what
happened once someone actually landed on the page, no pageviews, no sessions,
no engagement time, nothing. So any archetype that depends on "how well does
this page engage people once they arrive" just isn't answerable here. I'm
stuck inferring engagement indirectly through CTR, which is a search behavior
proxy, not a real on-site engagement measure.

**GSC coverage is only about 37%, and the missing 63% isn't random.** Content
without GSC data could mean a few different things, too new to have ranking
data yet, not indexed by Google at all, or just below whatever threshold
Search Console uses to report data. The data has no way to tell these
situations apart. So my archetypes only describe the ~37% slice that actually
has measurable search behavior. I can't say anything meaningful about the
other 63% of content, they're just invisible to this analysis, not "bad."

**The panel is growing, not fixed.** When I checked row counts per day, March 1
had about 276k rows and March 31 had about 331k. That's new content getting
added to the tracked set throughout the month, not the same set of pages
showing up every day. So when I compare early-March to late-March for trend,
some of that "difference" is really just newer content that wasn't around yet
on day one, not actual performance change.

**One month isn't enough to call something a real trend.** My trend_direction
feature is a slope calculated across March's 31 days, which is better than
just comparing two snapshots, but it's still only a month. A page could look
like it's "rising" just from normal week to week noise, or because of the
panel growth thing above, not because it's actually on a real upward trajectory.
To trust a trend label, I'd really want to see it hold up across two or three
months, not one.

**Nothing here tells you why.** Even if I find a clean group of pages that all
share a pattern, like low visibility but high CTR, the data can't tell me why
that's happening. Could be content quality, could be who's competing for that
keyword, could be a seasonal blip. Clustering shows me the pattern, not the
cause.

If I had to pick just one limitation to call out as the big one, it's probably
the growing panel issue, since it directly messes with the trend feature I'm
building, and I can actually point to the row counts as proof.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.